In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# ViT için hiperparametreler
patch_size = 16  # 16x16 boyutunda patch'ler
num_patches = (256 // patch_size) ** 2
projection_dim = 64
num_heads = 4
transformer_layers = 8
mlp_head_units = [2048, 1024]  # MLP baş katmanı boyutları

In [ ]:
class Patches(layers.Layer):
    def __init__(self, patch_size):
        super().__init__()
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

In [ ]:
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.num_patches = num_patches
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

In [ ]:
def create_vit_model(input_shape=(256, 256, 3), num_classes=3):
    inputs = layers.Input(shape=input_shape)
    
    # Patch oluşturma ve kodlama
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)

    # Transformer katmanları
    for _ in range(transformer_layers):
        # Layer Normalization
        x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
        
        # Multi-head attention
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=projection_dim, dropout=0.1
        )(x1, x1)
        
        # Skip connection 1
        x2 = layers.Add()([attention_output, encoded_patches])
        
        # Layer Normalization
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        
        # MLP
        x3 = mlp(x3, hidden_units=mlp_head_units, dropout_rate=0.1)
        
        # Skip connection 2
        encoded_patches = layers.Add()([x3, x2])

    # Sınıflandırma başı
    representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
    representation = layers.Flatten()(representation)
    representation = layers.Dropout(0.5)(representation)
    
    # MLP başı
    features = mlp(representation, hidden_units=mlp_head_units, dropout_rate=0.5)
    
    # Çıkış katmanı
    logits = layers.Dense(num_classes, activation="softmax")(features)
    
    return tf.keras.Model(inputs=inputs, outputs=logits)

def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x

In [ ]:
# Modeli oluştur
vit_model = create_vit_model()

# Optimizer
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-4)

# Modeli derle
vit_model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Callback'ler
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
]

# Modeli eğit
print("Vision Transformer eğitiliyor...")
vit_history = vit_model.fit(
    X_train_rgb, y_train,
    validation_data=(X_val_rgb, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks
)

In [ ]:
# Performans grafikleri
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(vit_history.history['accuracy'], label='Eğitim Doğruluğu')
plt.plot(vit_history.history['val_accuracy'], label='Doğrulama Doğruluğu')
plt.title('ViT Model Doğruluğu')
plt.ylabel('Doğruluk')
plt.xlabel('Epok')
plt.legend()

plt.subplot(1,2,2)
plt.plot(vit_history.history['loss'], label='Eğitim Kaybı')
plt.plot(vit_history.history['val_loss'], label='Doğrulama Kaybı')
plt.title('ViT Model Kaybı')
plt.ylabel('Kayıp')
plt.xlabel('Epok')
plt.legend()
plt.show()

# Test verisi üzerinde değerlendirme
test_loss, test_acc = vit_model.evaluate(X_val_rgb, y_val)
print(f"\nTest Doğruluğu: {test_acc:.4f}, Test Kaybı: {test_loss:.4f}")

In [ ]:
# Test görüntüleri üzerinde tahmin yapma
sample_idx = np.random.randint(len(X_val))
pred = vit_model.predict(X_val_rgb[sample_idx][np.newaxis, ...])

# Görselleştirme
plt.figure(figsize=(10,5))
plt.imshow(X_val[sample_idx], cmap='gray')
plt.title(f"Gerçek: {class_names[y_val[sample_idx]]}\nTahmin: {class_names[np.argmax(pred)]}")
plt.show()

print("Olasılıklar:")
for name, prob in zip(class_names, pred[0]):
    print(f"{name}: {prob:.4f}")